# <u>**[Hackathon] => Innomatics Research Labs | Advanced GenAI Internship | Entrance Test**</u>

# Dataset Overview and Problem Statement

## 1️⃣ Files Provided
You have three different data files, each in a different format, simulating real-world systems:

- 📂 **File 1:** `orders.csv` (Transactional Data) 
- 📂 **File 2:** `users.json` (User Master Data)   
- 📂 **File 3:** `restaurants.sql` (Restaurant Master Data)   

---

## 2️⃣ Step-by-Step: Combining the Datasets

**Step 1:** Load CSV Data (`orders.csv`)  
**Step 2:** Load JSON Data (`users.json`)  
**Step 3:** Load SQL Data (`restaurants.sql`)  
**Step 4:** Merge the Data  

Perform joins using keys:  

- `orders.user_id` → `users.user_id`  
- `orders.restaurant_id` → `restaurants.restaurant_id`  

**Join Type:** Left Join (to retain all orders)  

**Step 5:** Create Final Dataset  

The final dataset contains:  

- Order details  
- User information  
- Restaurant information  

**📁 Output File:** `final_food_delivery_dataset.csv`  

---

## 3️⃣ Final Dataset – What Students Should Understand

Students must analyze:  

- Order trends over time  
- User behavior patterns  
- City-wise and cuisine-wise performance  
- Membership impact (Gold vs Regular)  
- Revenue distribution and seasonality  

**📌 Note:** This final dataset is the only source of truth for all questions listed in this Hackathon.


# **1️⃣ Multiple Choice Questions (MCQs):-**

This section contains MCQs based on the given datasets. Analyze the data carefully and select the correct answers.

1. Which city has the highest total revenue (total_amount) from Gold members?  

2. Which cuisine has the highest average order value across all orders?  

3. How many distinct users placed orders worth more than ₹1000 in total (sum of all their orders)?  

4. Which restaurant rating range generated the highest total revenue?  

5. Among Gold members, which city has the highest average order value?  

6. Which cuisine has the lowest number of distinct restaurants but still contributes significant revenue?  

7. What percentage of total orders were placed by Gold members? (Rounded to nearest integer)  

8. Which restaurant has the highest average order value but less than 20 total orders?  

9. Which combination contributes the highest revenue?  

10. During which quarter of the year is the total revenue highest?  

---

In [51]:
import pandas as pd
import json
import re

# --- STEP 1: DATA LOADING & MERGING ---
orders_df = pd.read_csv('orders.csv')
orders_df['order_date'] = pd.to_datetime(orders_df['order_date'], dayfirst=True)

with open('users.json', 'r') as f:
    users_df = pd.DataFrame(json.load(f))

res_list = []
with open('restaurants.sql', 'r') as f:
    for line in f:
        if line.startswith('INSERT INTO restaurants VALUES'):
            match = re.search(r'\((.*)\);', line)
            if match:
                vals = match.group(1).split(', ')
                res_list.append({
                    'restaurant_id': int(vals[0]), 
                    'restaurant_name': vals[1].strip("'"),
                    'cuisine': vals[2].strip("'"), 
                    'rating': float(vals[3])
                })
restaurants_df = pd.DataFrame(res_list)

# Sabko merge karke ek master table banayein
df = pd.merge(orders_df, users_df, on='user_id', how='left')
df = pd.merge(df, restaurants_df, on='restaurant_id', how='left')

print("--- Master Data Ready ---\n")

--- Master Data Ready ---



In [9]:
# MCQ 1: Highest total revenue from Gold members by City
ans1 = df[df['membership'] == 'Gold'].groupby('city')['total_amount'].sum().idxmax()
print(f"Q1 (City): {ans1}")


Q1 (City): Chennai


In [11]:
# MCQ 2: Highest average order value by Cuisine
ans2 = df.groupby('cuisine')['total_amount'].mean().idxmax()
print(f"Q2 (Cuisine): {ans2}")

Q2 (Cuisine): Mexican


In [13]:
# MCQ 3: Users with total orders > 1000
user_sums = df.groupby('user_id')['total_amount'].sum()
ans3 = (user_sums > 1000).sum()
print(f"Q3 (Users Count): {ans3} (> 2000 range)")

Q3 (Users Count): 2544 (> 2000 range)


In [15]:
# MCQ 4: Revenue by Rating Range
def get_range(r):
    if 3.0 <= r <= 3.5: return '3.0 - 3.5'
    if 3.6 <= r <= 4.0: return '3.6 - 4.0'
    if 4.1 <= r <= 4.5: return '4.1 - 4.5'
    if 4.6 <= r <= 5.0: return '4.6 - 5.0'
df['rating_range'] = df['rating'].apply(get_range)
ans4 = df.groupby('rating_range')['total_amount'].sum().idxmax()
print(f"Q4 (Rating Range): {ans4}")

Q4 (Rating Range): 4.6 - 5.0


In [17]:
# MCQ 5: Highest average order value (Gold) by City
ans5 = df[df['membership'] == 'Gold'].groupby('city')['total_amount'].mean().idxmax()
print(f"Q5 (Gold Avg City): {ans5}")

Q5 (Gold Avg City): Chennai


In [19]:
# MCQ 6: Cuisine with lowest restaurants but high revenue
cuisine_counts = df.groupby('cuisine')['restaurant_id'].nunique()
ans6 = cuisine_counts.idxmin()
print(f"Q6 (Lowest Res Cuisine): {ans6}")

Q6 (Lowest Res Cuisine): Chinese


In [21]:
# MCQ 7: Percentage of Gold orders
gold_pct = (len(df[df['membership'] == 'Gold']) / len(df)) * 100
print(f"Q7 (Gold %): {round(gold_pct)}%")


Q7 (Gold %): 50%


In [56]:
# MCQ 8: Restaurant with highest average order value but < 20 orders

rest_stats = final_df.groupby('restaurant_name').agg(
    avg_order_value=('total_amount', 'mean'),
    order_count=('order_id', 'count')
).reset_index()

candidates = [
    'Grand Cafe Punjabi',
    'Grand Restaurant South Indian',
    'Ruchi Mess Multicuisine',
    'Ruchi Foods Chinese'
]

# Filter only given options AND < 20 orders
filtered = rest_stats[
    (rest_stats['restaurant_name'].isin(candidates)) &
    (rest_stats['order_count'] < 20)
]

if not filtered.empty:
    q8 = filtered.sort_values(
        by='avg_order_value',
        ascending=False
    ).iloc[0]['restaurant_name']
    print("Q8 Answer:", q8)
else:
    print("No restaurant found with less than 20 orders")


Q8 Answer: Ruchi Foods Chinese


In [25]:
# MCQ 9: Highest revenue combination (Membership + Cuisine)
options_q9 = [
    ('Gold', 'Indian'),
    ('Gold', 'Italian'),
    ('Regular', 'Indian'),
    ('Regular', 'Chinese')
]

# 2. In combinations ka revenue nikalein
combos = df.groupby(['membership', 'cuisine'])['total_amount'].sum()
q9_results = combos.loc[options_q9]

print("--- Question 9 Analysis ---")
print(q9_results)
print(f"\nHighest Revenue Combination: {q9_results.idxmax()}")

--- Question 9 Analysis ---
membership  cuisine
Gold        Indian      979312.31
            Italian    1005779.05
Regular     Indian      992100.27
            Chinese     952790.91
Name: total_amount, dtype: float64

Highest Revenue Combination: ('Gold', 'Italian')


In [27]:

# MCQ 10: Highest revenue quarter
df['quarter'] = df['order_date'].dt.quarter
ans10 = df.groupby('quarter')['total_amount'].sum().idxmax()
print(f"Q10 (Quarter): Q{ans10}")

Q10 (Quarter): Q3


# **2️⃣ Questions & Answers (Numerical):-**

This section contains questions with numerical answers. Please enter the correct values, including decimals wherever applicable.

1. How many total orders were placed by users with Gold membership?  

2. What is the total revenue (rounded to nearest integer) generated from orders placed in Hyderabad city?  

3. How many distinct users placed at least one order?  

4. What is the average order value (rounded to 2 decimals) for Gold members?  

5. How many orders were placed for restaurants with rating ≥ 4.5?  

6. How many orders were placed in the top revenue city among Gold members only?  

---


In [29]:
import pandas as pd
import json
import re

# 1. Orders CSV load 
orders_df = pd.read_csv('orders.csv')
orders_df['order_date'] = pd.to_datetime(orders_df['order_date'], dayfirst=True)

# 2. Users JSON load 
with open('users.json', 'r') as f:
    users_data = json.load(f)
users_df = pd.DataFrame(users_data)

# 3. Restaurants SQL parse 
restaurants_list = []
with open('restaurants.sql', 'r') as f:
    for line in f:
        if line.startswith('INSERT INTO restaurants VALUES'):
            match = re.search(r'\((.*)\);', line)
            if match:
                vals = match.group(1).split(', ')
                restaurants_list.append({
                    'restaurant_id': int(vals[0]), 
                    'cuisine': vals[2].strip("'"), 
                    'rating': float(vals[3])
                })
restaurants_df = pd.DataFrame(restaurants_list)

# 4. Saara data merge karna
orders_full = pd.merge(orders_df, users_df, on='user_id', how='left')
orders_full = pd.merge(orders_full, restaurants_df, on='restaurant_id', how='left')

In [31]:
# Q1
gold_orders_count = len(orders_full[orders_full['membership'] == 'Gold'])
print(f"Total Gold Membership Orders: {gold_orders_count}")


Total Gold Membership Orders: 4987


In [33]:
# Q2
hyd_revenue = round(orders_full[orders_full['city'] == 'Hyderabad']['total_amount'].sum())
print(f"Hyderabad Total Revenue: {hyd_revenue}")

Hyderabad Total Revenue: 1889367


In [35]:
# Q3
distinct_users = orders_df['user_id'].nunique()
print(f"Distinct Users: {distinct_users}")

Distinct Users: 2883


In [37]:
# Q4
gold_avg_value = round(orders_full[orders_full['membership'] == 'Gold']['total_amount'].mean(), 2)
print(f"Average Order Value (Gold): {gold_avg_value}")

Average Order Value (Gold): 797.15


In [39]:
# Q5
rating_45_orders = len(orders_full[orders_full['rating'] >= 4.5])
print(f"Orders with Restaurant Rating >= 4.5: {rating_45_orders}")

Orders with Restaurant Rating >= 4.5: 3374


In [41]:
# Q6
# Step A: Top revenue city for Gold members
top_gold_city = orders_full[orders_full['membership'] == 'Gold'].groupby('city')['total_amount'].sum().idxmax()

# Step B: Count orders for that city
top_gold_city_orders = len(orders_full[(orders_full['membership'] == 'Gold') & (orders_full['city'] == top_gold_city)])

print(f"Top Gold Revenue City: {top_gold_city}")
print(f"Orders in {top_gold_city}: {top_gold_city_orders}")

Top Gold Revenue City: Chennai
Orders in Chennai: 1337


# **3️⃣ Fill-in-the-Blanks:-**

This section contains fill-in-the-blank questions. Write the correct answer based on the given datasets.

1. The column used to join orders.csv and users.json is __________.  

2. The dataset containing cuisine and rating information is stored in __________ format.  

3. The total number of rows in the final merged dataset is __________.  

4. If a user has no matching record in users.json, the merged values will be __________.  

5. The Pandas function used to combine datasets based on a key is __________.  

6. The column membership in the final dataset originates from the __________ file.  

7. The join key used to combine orders data with restaurant details is __________.  

8. The column that helps identify the type of food served by a restaurant is __________.  

9. If a user places multiple orders, their personal details appear __________ times in the final merged dataset.  

In [47]:
import pandas as pd
import json
import re
import numpy as np

# Orders Dataset (Blank #1: Join Key 'user_id')
orders_df = pd.read_csv('orders.csv')

# Users Dataset (Blank #2: Format JSON, Blank #6: Membership column source)
with open('users.json', 'r') as f:
    users_df = pd.DataFrame(json.load(f))

# Restaurants Dataset (Blank #2: Format SQL, Blank #8: Cuisine column source)
res_list = []
with open('restaurants.sql', 'r') as f:
    for line in f:
        if line.startswith('INSERT INTO restaurants VALUES'):
            match = re.search(r'\((.*)\);', line)
            if match:
                vals = match.group(1).split(', ')
                res_list.append({
                    'restaurant_id': int(vals[0]), 
                    'cuisine': vals[2].strip("'")
                })
restaurants_df = pd.DataFrame(res_list)

# 2. Joining / Combining (Blank #5: pd.merge function)
# (Blank 1)
merged_step1 = pd.merge(orders_df, users_df, on='user_id', how='left')

# (Blank 7)
final_df = pd.merge(merged_step1, restaurants_df, on='restaurant_id', how='left')

# 3. VERIFICATION AND ANSWERS
print("--- FILL IN THE BLANKS: DATA-DRIVEN ANSWERS ---\n")

# Q1 Join Keys
print(f"1. Join key for Orders & Users: = user_id")


# Q2: Formats
print(f"2. Format of Cuisine/Rating data: = SQL (.sql)\n")

# Q3: Total Rows
print(f"3. Total rows in final merged dataset: = {len(final_df)}\n")

# Q4: Missing records behavior
print(f"4. Missing matching records are filled with: = NaN (Null)\n")

# Q5: Function used
print(f"5. Pandas function used to combine: = pd.merge()\n")

# Q6: Column Origins
print(f"6. Source of 'membership' column: = users.json\n")

# Q7: Join Keys
print(f"7. Join key for Orders & Restaurants: = restaurant_id\n")

# Q8: Column Origins
column_name = 'cuisine' 
source_file = 'restaurants.sql'
print(f"8. The column for food type is = '{column_name}' and its source is {source_file}\n")

# Q9: Multiple Orders Repetition
# Hum check karenge ki ek user jisne multiple orders diye hain, uske details repeat ho rahe hain ya nahi
user_counts = final_df['user_id'].value_counts()
top_user_orders = user_counts.iloc[0]
print(f"9. If a user places {top_user_orders} orders, their details appear: = {top_user_orders} times (Multiple times)")

--- FILL IN THE BLANKS: DATA-DRIVEN ANSWERS ---

1. Join key for Orders & Users: = user_id
2. Format of Cuisine/Rating data: = SQL (.sql)

3. Total rows in final merged dataset: = 10000

4. Missing matching records are filled with: = NaN (Null)

5. Pandas function used to combine: = pd.merge()

6. Source of 'membership' column: = users.json

7. Join key for Orders & Restaurants: = restaurant_id

8. The column for food type is = 'cuisine' and its source is restaurants.sql

9. If a user places 13 orders, their details appear: = 13 times (Multiple times)


# **END**